In [1]:
import pandas as pd

validated_misalignments = pd.read_json("../data/core/validated_misalignments.json")

In [2]:
# Calculate the valid rate for IDE and CLI sessions, respectively
validated_misalignments[
    validated_misalignments["session_type"].isin(["IDE", "CLI"])
].assign(is_valid=lambda df: df["label"].eq("VALID")).groupby("session_type")[
    "is_valid"
].mean()

session_type
CLI    0.534514
IDE    0.541682
Name: is_valid, dtype: float64

In [3]:
# Define the two bias types
NORMATIVE_PRIOR_BIAS = [
    "unrequested_action_without_pushback",
    "intention_claim_without_pushback",
    "collaboration_style_without_pushback",
    "evidence_contradicts_description",
]
OBSERVATIONAL_BLIND_SPOTS = [
    "invisible_project_context",
    "invisible_agent_action",
    "session_terminated_before_completion",
    "truncation",
]


def label_bias_type(category):
    if category in NORMATIVE_PRIOR_BIAS:
        return "Normative Prior Bias"
    if category in OBSERVATIONAL_BLIND_SPOTS:
        return "Observational Blind Spots"
    return None


# Filter to INVALID records from IDE/CLI sessions
invalid = validated_misalignments[
    validated_misalignments["session_type"].isin(["IDE", "CLI"])
    & validated_misalignments["label"].eq("INVALID")
].copy()
invalid["bias_type"] = invalid["invalid_category"].apply(label_bias_type)
invalid = invalid.dropna(subset=["bias_type"])


def format_count_pct(ct):
    """Return a DataFrame of 'count (row%)' strings."""
    row_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    return ct.astype(str) + " (" + row_pct.round(1).astype(str) + "%)"


# ---------- Bias type by session type (with overall row) ----------
ct_bias = pd.crosstab(invalid["session_type"], invalid["bias_type"])
ct_bias.loc["Overall"] = ct_bias.sum(axis=0)
display(format_count_pct(ct_bias))


# ---------- 8-category breakdown by session type (with overall column) ----------
ct_full = pd.crosstab(invalid["session_type"], invalid["invalid_category"])
row_pct_full = (ct_full.div(ct_full.sum(axis=1), axis=0) * 100).round(2)
overall_pct = (ct_full.sum(axis=0) / ct_full.values.sum() * 100).round(2)

category_table = pd.DataFrame(
    {
        "ide_pct_within_invalid": row_pct_full.loc["IDE"],
        "cli_pct_within_invalid": row_pct_full.loc["CLI"],
        "overall_pct_within_invalid": overall_pct,
        "bias_type": [label_bias_type(c) for c in ct_full.columns],
    }
).sort_values("overall_pct_within_invalid", ascending=False)
display(category_table)

bias_type,Normative Prior Bias,Observational Blind Spots
session_type,,
CLI,1773 (35.9%),3160 (64.1%)
IDE,4046 (45.8%),4784 (54.2%)
Overall,5819 (42.3%),7944 (57.7%)


,ide_pct_within_invalid,cli_pct_within_invalid,overall_pct_within_invalid,bias_type
invalid_category,,,,
invisible_agent_action,28.37,48.75,35.68,Observational Blind Spots
intention_claim_without_pushback,21.30,16.54,19.60,Normative Prior Bias
unrequested_action_without_pushback,16.24,12.69,14.97,Normative Prior Bias
session_terminated_before_completion,15.62,8.21,12.96,Observational Blind Spots
invisible_project_context,9.74,6.85,8.70,Observational Blind Spots
collaboration_style_without_pushback,4.99,2.92,4.25,Normative Prior Bias
evidence_contradicts_description,3.28,3.79,3.47,Normative Prior Bias
truncation,0.45,0.24,0.38,Observational Blind Spots
